# Artificial Intelligence – Agents Laboratory Exercise
## Constructing a Goal-Based Agent Using a Large Language Model

**Course:** CSF407 / Artificial Intelligence Laboratory  
**Topic:** Goal-Based Agents, State Space Search, and LLM-Assisted Engineering  
**Date:** September 30, 2026  

---
## Executive Summary
This laboratory focuses on the conceptual design, formal specification, and implementation of an intelligent **goal-based agent** navigating an autonomous warehouse environment. Using Large Language Model (LLM) collaboration, we explore:
1. The defining properties of goal-based agents versus simple reflex agents.
2. The PEAS specification (Performance measure, Environment, Actuators, Sensors) and environment taxonomy.
3. State-space search representation and Breadth-First Search (BFS) path optimality on unweighted grids.
4. Iterative prompt engineering and verification methodologies.


---
## Task 1: Understanding the Problem

### 1. What is the environment?
The environment is a discrete 2-dimensional warehouse grid measuring 7 rows by 21 columns (147 total grid cells).
Under the standard Russell & Norvig environment taxonomy:
- **Fully Observable:** The agent has complete access to the warehouse map, knowing obstacle coordinates, its current position, and the destination.
- **Static:** The warehouse map, shelf obstacles, start point, and goal do not change while the agent deliberates.
- **Deterministic:** Every move action in a cardinal direction results in the exact intended displacement with probability $1.0$.
- **Discrete:** State coordinates $(r, c)$ and action steps occur in discrete integer increments.
- **Single-Agent:** The vehicle operates alone without other dynamic vehicles or human interference.

### 2. What is the goal of the agent?
The agent's goal is to transport a package from the loading bay start position $S = (1, 1)$ to the dispatch area destination $G = (1, 19)$ along a collision-free path that traverses only free space (`.`) and never intersects shelving obstacles (`#`), while minimizing path length (number of moves).

### 3. What actions are available to the agent?
The vehicle has four deterministic cardinal movement actions:
- `Up`: $(r - 1, c)$
- `Down`: $(r + 1, c)$
- `Left`: $(r, c - 1)$
- `Right`: $(r, c + 1)$
Each action is valid if and only if the destination square is within map boundaries and is not an obstacle.

### 4. What information must the agent maintain in order to choose its next action?
To make decisions, the agent must maintain:
1. **Model of the environment:** The grid dimensions and locations of all static obstacle blocks (`#`).
2. **Current state:** The vehicle's current coordinates $(r, c)$.
3. **Goal state:** The coordinates of the dispatch area $G = (1, 19)$.
4. **Transition model:** Knowledge of how actions update coordinates ($s' = 	ext{Result}(s, a)$).
5. **Search / Planning state:** A sequence of planned actions (or a frontier/visited set during search) to look ahead from start to goal.

### 5. Why is this an example of a goal-based agent rather than a simple reflex agent?
- A **simple reflex agent** selects actions based solely on the current percept via condition-action rules (e.g., `if front is blocked, turn right`). Because it cannot look ahead or evaluate future consequences, it has no memory of past decisions or knowledge of a destination. In a maze-like warehouse with concave obstacles, dead-ends, and detours, a reflex agent easily becomes trapped in infinite loops.
- A **goal-based agent** explicitly incorporates a representation of its desired goal ($G$) and deliberates over sequences of future actions using search and planning. It selects actions specifically because they form part of a sequence that reaches the goal state.

---
### Think About It: Scaling to a Warehouse Twice as Large
> **Question:** Suppose the warehouse becomes twice as large. Would the same search strategy still be appropriate? What additional difficulties might arise?

**Analytical Insight:**
1. **Search Algorithm Scalability:** Breadth-First Search (BFS) is optimal for unweighted graphs, but its time and space complexity are $\mathcal{O}(b^d)$, where $b \approx 3$ is the branching factor and $d$ is the path depth. For a moderately doubled grid ($14 \times 42 = 588$ states), BFS remains fast. However, as dimensions scale to tens of thousands of cells, uninformed BFS expands a circular wave of nodes in all directions, wasting memory and CPU cycles expanding states that lead away from the goal.
2. **Heuristic Search:** In large grids, an **informed search algorithm like $A^*$ Search** with a consistent heuristic (such as Manhattan distance $h(n) = |r_n - r_G| + |c_n - c_G|$) is vastly superior. $A^*$ directs exploration towards the goal, expanding significantly fewer nodes while retaining optimality.
3. **Dynamic Warehouse Challenges:** In real industrial warehouses, dynamic obstacles (other automated guided vehicles, human pickers, temporary pallet drop-offs) violate the static environment assumption, necessitating real-time replanning (e.g., Lifelong Planning $A^*$ or $D^*$ Lite) rather than purely offline pre-computation.


---
## Task 2: Designing the Agent

### Agent Components
1. **Environment:** 2D discrete grid with shelving units.
2. **Current State:** $(r, c) \in [0, 6] \times [0, 20]$.
3. **Goal State:** $(1, 19)$.
4. **Available Actions:** $\{\text{Up}, \text{Down}, \text{Left}, \text{Right}\}$.
5. **Decision-Making Component:** Breadth-First Search (BFS) path planning algorithm over the state space graph.

### Goal-Based Agent Block Diagram
The agent conforms to the standard Russell & Norvig Goal-Based Agent Architecture:
```text
                    +------------------------------------+
                    |            ENVIRONMENT             |
                    |          (Warehouse Grid)          |
                    +-----------------+------------------+
                                      |
                                      | Percepts (Coordinates & Map)
                                      v
                             +-----------------+
                             |     SENSORS     |
                             +--------+--------+
                                      |
                                      v
                             +-----------------+
                             |  CURRENT STATE  | <-----+
                             | (Current r, c)  |       |
                             +--------+--------+       |
                                      |                | Updates
                                      v                | State
+--------------------+       +-----------------+       |
|  TRANSITION MODEL  | ----> |  WHAT WILL IT   |       |
| (How actions work) |       |  BE LIKE IF I   |       |
+--------------------+       |   DO ACTION A?  |       |
                             +--------+--------+       |
                                      |                |
                                      v                |
+--------------------+       +-----------------+       |
|    GOAL (G)        | ----> | DECISION MAKER: |       |
| (Dispatch location)|       | BFS PATH PLANNER|       |
+--------------------+       +--------+--------+       |
                                      |                |
                                      v                |
                             +-----------------+       |
                             | WHAT ACTION TO  | ------+
                             |   TAKE NOW?     |
                             +--------+--------+
                                      |
                                      v
                             +-----------------+
                             |    ACTUATORS    |
                             +--------+--------+
                                      |
                                      | Move (Up/Down/Left/Right)
                                      v
                    +------------------------------------+
                    |            ENVIRONMENT             |
                    +------------------------------------+
```


---
## Task 3: Implementation, Execution, and Verification

### Prompt Provided to the LLM
```text
Write a well-documented Python program implementing a goal-based agent for the warehouse navigation problem.
The program should:
- Represent the warehouse as a two-dimensional grid;
- Determine a collision-free path from S to G;
- Avoid all obstacles;
- Print either the path found or a suitable message if no path exists;
- Explain the search algorithm that has been chosen and why it is appropriate.
```


In [1]:
from collections import deque
import numpy as np

# 1. Environment Definition
WAREHOUSE_MAP = [
    "#####################",
    "#S....#............G#",
    "#.##....##########..#",
    "#....##.............#",
    "#.######.###.#.###..#",
    "#........#..........#",
    "#####################",
]

ACTIONS = {
    "Up":    (-1, 0),
    "Down":  (1, 0),
    "Left":  (0, -1),
    "Right": (0, 1),
}

def find_symbol(grid, symbol):
    for r, row in enumerate(grid):
        for c, cell in enumerate(row):
            if cell == symbol:
                return (r, c)
    raise ValueError(f"Symbol {symbol!r} not found in warehouse map")

def in_bounds(grid, r, c):
    return 0 <= r < len(grid) and 0 <= c < len(grid[0])

def is_free(grid, r, c):
    return grid[r][c] != "#"

class WarehouseAgent:
    def __init__(self, grid):
        self.grid = grid
        self.state = find_symbol(grid, "S")
        self.goal = find_symbol(grid, "G")

    def search(self):
        start, goal = self.state, self.goal
        frontier = deque([start])
        came_from = {start: None}
        action_taken = {start: None}
        nodes_expanded = 0

        while frontier:
            current = frontier.popleft()
            nodes_expanded += 1

            if current == goal:
                return self._reconstruct(came_from, action_taken, goal), nodes_expanded

            for action, (dr, dc) in ACTIONS.items():
                nr, nc = current[0] + dr, current[1] + dc
                neighbor = (nr, nc)

                if in_bounds(self.grid, nr, nc) and is_free(self.grid, nr, nc):
                    if neighbor not in came_from:
                        came_from[neighbor] = current
                        action_taken[neighbor] = action
                        frontier.append(neighbor)

        return (None, None), nodes_expanded

    @staticmethod
    def _reconstruct(came_from, action_taken, goal):
        path, actions = [], []
        node = goal
        while node is not None:
            path.append(node)
            if action_taken[node] is not None:
                actions.append(action_taken[node])
            node = came_from[node]
        path.reverse()
        actions.reverse()
        return path, actions

    def run(self):
        print(f"Start state S: {self.state}")
        print(f"Goal state  G: {self.goal}\n")
        
        (path, actions), nodes_expanded = self.search()
        
        if path is None:
            print("No collision-free path exists.")
            return

        print(f"Path successfully found!")
        print(f"Nodes expanded: {nodes_expanded}")
        print(f"Path length:    {len(path)} cells ({len(actions)} move actions)")
        print("\nSequence of coordinates (row, col):")
        print("  " + " -> ".join(str(p) for p in path))
        print("\nSequence of actions:")
        print("  " + ", ".join(actions))
        
        # Grid overlay visualization
        display_grid = [list(row) for row in self.grid]
        for r, c in path:
            if display_grid[r][c] not in ('S', 'G'):
                display_grid[r][c] = '*'
        print("\nTrajectory Visualized on Grid ('*' = path):")
        for row in display_grid:
            print("  " + "".join(row))

# Run the agent
agent = WarehouseAgent(WAREHOUSE_MAP)
agent.run()


Start state S: (1, 1)
Goal state  G: (1, 19)

Path successfully found!
Nodes expanded: 59
Path length:    21 cells (20 move actions)

Sequence of coordinates (row, col):
  (1, 1) -> (1, 2) -> (1, 3) -> (1, 4) -> (2, 4) -> (2, 5) -> (2, 6) -> (2, 7) -> (1, 7) -> (1, 8) -> (1, 9) -> (1, 10) -> (1, 11) -> (1, 12) -> (1, 13) -> (1, 14) -> (1, 15) -> (1, 16) -> (1, 17) -> (1, 18) -> (1, 19)

Sequence of actions:
  Right, Right, Right, Down, Right, Right, Right, Up, Right, Right, Right, Right, Right, Right, Right, Right, Right, Right, Right, Right

Trajectory Visualized on Grid ('*' = path):
  #####################
  #S***.#************G#
  #.##****##########..#
  #....##.............#
  #.######.###.#.###..#
  #........#..........#
  #####################


---
## Task 3: Evaluation Questions & Reflection

### 1. Did the LLM generate a working program on the first attempt?
**Yes.** The program compiled and ran flawlessly on the first attempt. It accurately parsed the 2D grid representation, correctly located start `S = (1, 1)` and goal `G = (1, 19)`, enforced non-crossing constraint checks on `#` obstacles, and executed a clean Breadth-First Search (BFS) returning the optimal 20-move path.

### 2. If not, how can you improve your prompt?
Although the initial code functioned correctly, prompt engineering can be refined for greater robustness and extensibility by:
- Explicitly providing the grid representation data structure and boundary constraints.
- Requiring unit test assertions (e.g. confirming no coordinate in the path contains `#` and that start and end positions match).
- Requesting step-by-step telemetry, such as recording the number of nodes expanded and the memory usage of the frontier.
- Requesting algorithmic comparisons (e.g. comparing BFS against A* search with Manhattan distance).

### 3. What search algorithm did the LLM choose?
The LLM selected **Breadth-First Search (BFS)** using a double-ended queue (`collections.deque`).

### 4. Why do you think the LLM selected this algorithm?
The LLM selected BFS because:
1. **Unweighted Step Costs:** Every move in the grid costs exactly 1 unit of distance. On graphs with uniform step costs, BFS is guaranteed to be **complete** and **optimal** (it finds the path with the minimum number of moves).
2. **Guaranteed Termination on Finite Graphs:** By maintaining a `came_from` dictionary (visited set), BFS avoids cyclic paths and terminates in finite time $\mathcal{O}(V + E)$.
3. **Implementation Simplicity:** BFS avoids the overhead of managing a priority queue or defining domain-specific heuristic functions required by $A^*$. For a 147-cell grid, the time and memory differences between BFS and $A^*$ are completely negligible.

---
## Summary of Key Takeaways
- **Goal-Based Deliberation:** Goal-based agents decouple *what to achieve* (the goal condition) from *how to behave* (the search procedure), offering adaptability that reflex agents lack.
- **Search as Deliberation:** Search algorithms simulate actions ahead in time within an internal world model, preventing irreversible real-world mistakes (like driving into a warehouse shelf).
- **LLM as Engineering Collaborator:** The LLM accelerates code implementation and documentation, while human engineering oversight remains vital to specify exact constraints, verify edge cases, and validate optimality.
